In [ ]:
# %pip install docling pymupdf python-docx

  Using cached lxml-6.0.2-cp313-cp313-macosx_10_13_universal2.whl.metadata (3.6 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 378.9 kB/s  0:00:09 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 449.5 kB/s  0:00:11 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 418.5 kB/s  0:00:07m0:00:0100:01
Using cached lxml-6.0.2-cp313-cp313-macosx_10_13_universal2.whl (8.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 439.0 kB/s  0:00:05 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 399.7 kB/s  0:00:01m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 661.5 kB/s  0:00:31m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 872.6 kB/s  0:00:05 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 629.3 kB/s  0:00:15m0:00:0100:01
Using cached req

In [9]:
# Optional: useful later for embeddings and chunking
# %pip install langchain tiktoken

In [10]:
from pathlib import Path
import os

# Document processing
import fitz  # PyMuPDF
import docx  # python-docx

# Optional (if using Docling later)
from docling.document_converter import DocumentConverter

/Users/ramy/Desktop/research_project/research_pro_repo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
DATA_FOLDER = Path("DataSource")

SUPPORTED_EXTENSIONS = {".pdf", ".doc", ".docx"}

documents = []

for file in DATA_FOLDER.rglob("*"):
    if file.suffix.lower() in SUPPORTED_EXTENSIONS:
        documents.append(file)

print(f"Found {len(documents)} documents\n")

for doc in documents:
    print(doc)

Found 15 documents

DataSource/Ramani_Krackov_2012.pdf
DataSource/Van de Ridder_Wijnen-Meijer_2023.pdf
DataSource/Van de Ridder 2023_Pendletons Rules a Feedback Method.pdf
DataSource/Hewson_Little_1998_giving feedback in medical education.pdf
DataSource/Hattie and Timperley_2007.pdf
DataSource/2026-03-12_Allgemeines_Feedbackregeln.docx
DataSource/Fabry_Handbuch Feedback_Lehrende.pdf
DataSource/Gigante-J_Getting-Beyond-Good-Job_How-to-Give-Effective-Feedback_Pediatrics-2011.pdf
DataSource/Wisniewsi_Zierer_Hattie_2020.pdf
DataSource/MEDIC_FLYER_Mentoring_E11_220630_Ansicht.pdf
DataSource/MEDiC_2023_Leitfaden_1-1_Mentoring-aktuell.pdf
DataSource/Kunz et al_2019.pdf
DataSource/Poster_Feedback_MITZ.pdf
DataSource/Hillienhof_2018.pdf
DataSource/Thrien_Feedback_2020.pdf


In [13]:
doc_records = []

for file in DATA_FOLDER.rglob("*"):
    if file.suffix.lower() in SUPPORTED_EXTENSIONS:
        doc_records.append({
            "path": str(file),
            "name": file.name,
            "type": file.suffix.lower()
        })

print(f"Loaded {len(doc_records)} documents")
doc_records[:5]

Loaded 15 documents


[{'path': 'DataSource/Ramani_Krackov_2012.pdf',
  'name': 'Ramani_Krackov_2012.pdf',
  'type': '.pdf'},
 {'path': 'DataSource/Van de Ridder_Wijnen-Meijer_2023.pdf',
  'name': 'Van de Ridder_Wijnen-Meijer_2023.pdf',
  'type': '.pdf'},
 {'path': 'DataSource/Van de Ridder 2023_Pendletons Rules a Feedback Method.pdf',
  'name': 'Van de Ridder 2023_Pendletons Rules a Feedback Method.pdf',
  'type': '.pdf'},
 {'path': 'DataSource/Hewson_Little_1998_giving feedback in medical education.pdf',
  'name': 'Hewson_Little_1998_giving feedback in medical education.pdf',
  'type': '.pdf'},
 {'path': 'DataSource/Hattie and Timperley_2007.pdf',
  'name': 'Hattie and Timperley_2007.pdf',
  'type': '.pdf'}]

In [14]:
from pathlib import Path
import json
import fitz
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

extracted_docs = []
unsupported_docs = []
failed_docs = []

for record in doc_records:
    file_path = Path(record["path"])
    suffix = file_path.suffix.lower()

    try:
        # Legacy .doc is not handled here
        if suffix == ".doc":
            unsupported_docs.append({
                "path": str(file_path),
                "name": file_path.name,
                "type": suffix,
                "reason": "Legacy .doc is not supported in this pipeline. Convert it to .docx first."
            })
            continue

        # Preferred unified path: Docling
        if suffix in {".pdf", ".docx"}:
            result = converter.convert(file_path)
            doc_obj = result.document

            # Prefer markdown for RAG because headings/lists survive better
            text = doc_obj.export_to_markdown()

            extracted_docs.append({
                "path": str(file_path),
                "name": file_path.name,
                "type": suffix,
                "text": text,
                "source_method": "docling"
            })
            continue

    except Exception as e:
        # PDF fallback to PyMuPDF
        if suffix == ".pdf":
            try:
                pdf = fitz.open(file_path)
                pages = []

                for page_num, page in enumerate(pdf, start=1):
                    page_text = page.get_text("text", sort=True)
                    pages.append(f"\n\n--- Page {page_num} ---\n{page_text}")

                extracted_docs.append({
                    "path": str(file_path),
                    "name": file_path.name,
                    "type": suffix,
                    "text": "".join(pages),
                    "source_method": "pymupdf_fallback"
                })
            except Exception as inner_e:
                failed_docs.append({
                    "path": str(file_path),
                    "name": file_path.name,
                    "type": suffix,
                    "error": str(inner_e)
                })
        else:
            failed_docs.append({
                "path": str(file_path),
                "name": file_path.name,
                "type": suffix,
                "error": str(e)
            })

print(f"Extracted successfully: {len(extracted_docs)}")
print(f"Unsupported (.doc): {len(unsupported_docs)}")
print(f"Failed: {len(failed_docs)}")

if unsupported_docs:
    print("\nUnsupported files:")
    for item in unsupported_docs:
        print("-", item["name"])

if failed_docs:
    print("\nFailed files:")
    for item in failed_docs:
        print("-", item["name"], "->", item["error"])

Extracted successfully: 15
Unsupported (.doc): 0
Failed: 0


In [19]:
import re
import unicodedata

def normalize_text(text: str) -> str:
    """
    Clean extracted text while preserving paragraph boundaries.
    Fix common PDF unicode artifacts.
    """
    if not text:
        return ""

    # Normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # Remove soft hyphens
    text = text.replace("\u00ad", "")

    # Replace non-breaking space with normal space
    text = text.replace("\u00a0", " ")

    # Replace other strange PDF spacing characters
    weird_spaces = [
        "\u2000","\u2001","\u2002","\u2003","\u2004",
        "\u2005","\u2006","\u2007","\u2008","\u2009",
        "\u200a","\u202f","\u205f","\u3000"
    ]
    for ws in weird_spaces:
        text = text.replace(ws, " ")

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove trailing spaces
    text = "\n".join(line.rstrip() for line in text.splitlines())

    # Collapse repeated spaces
    text = re.sub(r"[ \t]{2,}", " ", text)

    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


normalized_docs = []

for doc in extracted_docs:
    clean_text = normalize_text(doc["text"])

    normalized_docs.append({
        "source": doc["name"],
        "path": doc["path"],
        "file_type": doc["type"],
        "source_method": doc["source_method"],
        "text": clean_text,
        "char_count": len(clean_text)
    })

print(f"Normalized documents: {len(normalized_docs)}")
normalized_docs[:2]

Normalized documents: 15


[{'source': 'Ramani_Krackov_2012.pdf',
  'path': 'DataSource/Ramani_Krackov_2012.pdf',
  'file_type': '.pdf',
  'source_method': 'docling',
  'text': "## TWELVE TIPS\n\n## Twelve tips for giving feedback effectively in the clinical environment\n\nSUBHA RAMANI 1 &amp; SHARON K. KRACKOV 2 1 Harvard Medical School, USA, 2 Albany Medical College, USA\n\n## Abstract\n\nBackground: Feedback is an essential element of the educational process for clinical trainees. Performance-based feedback enables good habits to be reinforced and faulty ones to be corrected. Despite its importance, most trainees feel that they do not receive adequate feedback and if they do, the process is not effective.\n\nAims and methods: The authors reviewed the literature on feedback and present the following 12 tips for clinical teachers to provide effective feedback to undergraduate and graduate medical trainees. In most of the tips, the focus is the individual teacher in clinical settings, although some of the sugges

In [20]:
# ---- Debug document content before chunking ----

# show available documents
print("Available documents:\n")

for i, doc in enumerate(normalized_docs):
    print(f"{i} -> {doc['source']} ({doc['char_count']} characters)")


# choose document index
doc_index = 9   # change this number to inspect another file


# preview length
preview_chars = 3000


doc = normalized_docs[doc_index]

print("\n" + "="*80)
print("SOURCE:", doc["source"])
print("TYPE:", doc["file_type"])
print("METHOD:", doc["source_method"])
print("CHAR COUNT:", doc["char_count"])
print("="*80)

print(doc["text"][:preview_chars])

print("\n\n--- END OF PREVIEW ---")

Available documents:

0 -> Ramani_Krackov_2012.pdf (23549 characters)
1 -> Van de Ridder_Wijnen-Meijer_2023.pdf (16503 characters)
2 -> Van de Ridder 2023_Pendletons Rules a Feedback Method.pdf (16503 characters)
3 -> Hewson_Little_1998_giving feedback in medical education.pdf (29562 characters)
4 -> Hattie and Timperley_2007.pdf (107280 characters)
5 -> 2026-03-12_Allgemeines_Feedbackregeln.docx (3435 characters)
6 -> Fabry_Handbuch Feedback_Lehrende.pdf (77216 characters)
7 -> Gigante-J_Getting-Beyond-Good-Job_How-to-Give-Effective-Feedback_Pediatrics-2011.pdf (15663 characters)
8 -> Wisniewsi_Zierer_Hattie_2020.pdf (64908 characters)
9 -> MEDIC_FLYER_Mentoring_E11_220630_Ansicht.pdf (6800 characters)
10 -> MEDiC_2023_Leitfaden_1-1_Mentoring-aktuell.pdf (26280 characters)
11 -> Kunz et al_2019.pdf (63213 characters)
12 -> Poster_Feedback_MITZ.pdf (1066 characters)
13 -> Hillienhof_2018.pdf (3929 characters)
14 -> Thrien_Feedback_2020.pdf (44438 characters)

SOURCE: MEDIC_FLYER_Mentor

In [21]:
doc_index = 9
print(normalized_docs[doc_index]["text"][:2000])

## WasistMentoring?

John Maxwell sagte einmal:

'Wenn Du schnell sein willst, geh' Deinen Weg alleine. Wenn Du weit kommen willst, nimm andere mit.'

In diesem Zitat liegt eine der wesentlichen Erkenntnisse, warum Mentoring ein so wichtiges Instrument für die persönlicheundberuflicheWeiterentwicklungist. Mentoring ist ein branchenübergreifendes Instrument, das zur gezielten Förderung von Fach- und Führungskräften eingesetzt wird.DurcheinenpersönlicheninformellenAustausch zwischen einem erfahrenen Ratgeber (Mentor:in) und einer Nachwuchskraft (Mentee) soll die Entwicklung individueller Kompetenzen und Fähigkeiten der:s Mentees direkt gefördert unddadurchdiespäterenberuflichenEin-undAufstiegschancenverbessertwerden.

## 1:1MEDiC-Mentoring

## Klinisches Mentoring

## Wissenschaftliches Mentoring

## Förderung und Reflexion

- praxisorientierter ärztlicher Kompetenzen,

· derberuflichenOrientierung,

- interprofessioneller und interdisziplinärer Zusammenarbeit,
- von Leitungs- und Manage

In [22]:
from pathlib import Path
import fitz
from docling.document_converter import DocumentConverter

doc_index = 9
file_path = Path(normalized_docs[doc_index]["path"])

print("FILE:", file_path)

# ---- Docling extraction ----
converter = DocumentConverter()
docling_result = converter.convert(file_path)
docling_text = docling_result.document.export_to_markdown()

# ---- PyMuPDF extraction ----
pdf = fitz.open(file_path)
pymupdf_pages = []
for page_num, page in enumerate(pdf, start=1):
    page_text = page.get_text("text", sort=True)
    pymupdf_pages.append(f"\n\n--- Page {page_num} ---\n{page_text}")
pymupdf_text = "".join(pymupdf_pages)

print("\n" + "=" * 80)
print("DOCLING PREVIEW")
print("=" * 80)
print(docling_text[:2000])

print("\n" + "=" * 80)
print("PYMUPDF PREVIEW")
print("=" * 80)
print(pymupdf_text[:2000])

FILE: DataSource/MEDIC_FLYER_Mentoring_E11_220630_Ansicht.pdf

DOCLING PREVIEW
## Was­ist­Mentoring?

John Maxwell sagte einmal:

'Wenn Du schnell sein willst, geh' Deinen Weg alleine. Wenn Du weit kommen willst, nimm andere mit.'

In diesem Zitat liegt eine der wesentlichen   Erkenntnisse, warum Mentoring ein so wichtiges Instrument für die ­ persönliche­und­berufliche­Weiterentwicklung­ist.­­ Mentoring­ ist ein branchenübergreifendes Instrument, das zur gezielten Förderung von Fach- und Führungskräften eingesetzt wird.­Durch­einen­persönlichen­informellen­Austausch­ zwischen   einem erfahrenen Ratgeber (Mentor:in) und einer Nachwuchskraft (Mentee) soll die Entwicklung individueller Kompetenzen und Fähigkeiten der:s Mentees direkt gefördert und­dadurch­die­späteren­beruflichen­Ein-­und­Aufstiegschancen­verbessert­werden.

## 1:1­MEDiC-Mentoring

## Klinisches Mentoring

## Wissenschaftliches Mentoring

## Förderung und Reflexion

-   praxisorientierter ärztlicher Kompetenzen,

·­­ der

In [23]:
print("\n" + "=" * 80)
print("PYMUPDF PREVIEW")
print("=" * 80)
print(pymupdf_text[:2000])


PYMUPDF PREVIEW


--- Page 1 ---
Was ist Mentoring?                            Organisation
                John Maxwell sagte einmal:
   „Wenn Du schnell sein willst, geh‘ Deinen
        Weg alleine. Wenn Du weit                         Medizincampus
     kommen willst, nimm andere mit.“                      Chemnitz der TU Dresden


 In diesem Zitat liegt eine der wesentlichen ­Erkenntnisse,
­warum Mentoring ein so wichtiges Instrument für die
 ­persönliche und berufliche Weiterentwicklung ist. ­Mentoring
 ist ein branchenübergreifendes Instrument, das zur geziel-                           Kontaktten Förderung von Fach- und Führungskräften eingesetzt
 wird. Durch einen persönlichen informellen Austausch                                                              Mentoringkoordination
zwischen ­einem erfahrenen Ratgeber (Mentor:in) und einer
                                                           Peggy GierschnerNachwuchskraft (Mentee) soll die Entwicklung individueller
         

In [24]:
print("\n" + "=" * 80)
print("DOCLING PREVIEW")
print("=" * 80)
print(docling_text[:2000])


DOCLING PREVIEW
## Was­ist­Mentoring?

John Maxwell sagte einmal:

'Wenn Du schnell sein willst, geh' Deinen Weg alleine. Wenn Du weit kommen willst, nimm andere mit.'

In diesem Zitat liegt eine der wesentlichen   Erkenntnisse, warum Mentoring ein so wichtiges Instrument für die ­ persönliche­und­berufliche­Weiterentwicklung­ist.­­ Mentoring­ ist ein branchenübergreifendes Instrument, das zur gezielten Förderung von Fach- und Führungskräften eingesetzt wird.­Durch­einen­persönlichen­informellen­Austausch­ zwischen   einem erfahrenen Ratgeber (Mentor:in) und einer Nachwuchskraft (Mentee) soll die Entwicklung individueller Kompetenzen und Fähigkeiten der:s Mentees direkt gefördert und­dadurch­die­späteren­beruflichen­Ein-­und­Aufstiegschancen­verbessert­werden.

## 1:1­MEDiC-Mentoring

## Klinisches Mentoring

## Wissenschaftliches Mentoring

## Förderung und Reflexion

-   praxisorientierter ärztlicher Kompetenzen,

·­­ der­beruflichen­Orientierung,

-   interprofessioneller und inter

In [25]:
from pathlib import Path
import fitz
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

extracted_docs = []
failed_docs = []

for record in doc_records:
    file_path = Path(record["path"])
    suffix = file_path.suffix.lower()

    try:
        if suffix == ".pdf":
            pdf = fitz.open(file_path)
            pages = []

            for page_num, page in enumerate(pdf, start=1):
                page_text = page.get_text("text", sort=True)
                pages.append(f"\n\n--- Page {page_num} ---\n{page_text}")

            text = "".join(pages)

            extracted_docs.append({
                "path": str(file_path),
                "name": file_path.name,
                "type": suffix,
                "text": text,
                "source_method": "pymupdf"
            })

        elif suffix == ".docx":
            result = converter.convert(file_path)
            text = result.document.export_to_markdown()

            extracted_docs.append({
                "path": str(file_path),
                "name": file_path.name,
                "type": suffix,
                "text": text,
                "source_method": "docling"
            })

    except Exception as e:
        failed_docs.append({
            "path": str(file_path),
            "name": file_path.name,
            "type": suffix,
            "error": str(e)
        })

print(f"Extracted successfully: {len(extracted_docs)}")
print(f"Failed: {len(failed_docs)}")

if failed_docs:
    for item in failed_docs:
        print(item)

Extracted successfully: 15
Failed: 0


In [26]:
import re
import unicodedata

def normalize_text(text: str) -> str:
    """
    Clean extracted text while preserving paragraph boundaries.
    Fix common PDF unicode and line-break artifacts.
    """
    if not text:
        return ""

    text = unicodedata.normalize("NFKC", text)

    # Common PDF artifacts
    text = text.replace("\u00ad", "")   # soft hyphen
    text = text.replace("\u00a0", " ")  # non-breaking space

    weird_spaces = [
        "\u2000", "\u2001", "\u2002", "\u2003", "\u2004",
        "\u2005", "\u2006", "\u2007", "\u2008", "\u2009",
        "\u200a", "\u202f", "\u205f", "\u3000"
    ]
    for ws in weird_spaces:
        text = text.replace(ws, " ")

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Join words broken across lines by hyphenation
    # Example: Entwick-
    #          lung -> Entwicklung
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Trim trailing spaces line by line
    text = "\n".join(line.rstrip() for line in text.splitlines())

    # Turn single newlines into spaces, preserve paragraph breaks
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Collapse extra spaces / tabs
    text = re.sub(r"[ \t]{2,}", " ", text)

    # Collapse too many blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


normalized_docs = []

for doc in extracted_docs:
    clean_text = normalize_text(doc["text"])

    normalized_docs.append({
        "source": doc["name"],
        "path": doc["path"],
        "file_type": doc["type"],
        "source_method": doc["source_method"],
        "text": clean_text,
        "char_count": len(clean_text)
    })

print(f"Normalized documents: {len(normalized_docs)}")
normalized_docs[:2]

Normalized documents: 15


[{'source': 'Ramani_Krackov_2012.pdf',
  'path': 'DataSource/Ramani_Krackov_2012.pdf',
  'file_type': '.pdf',
  'source_method': 'pymupdf',
  'text': "--- Page 1 --- 2012; 34: 787–791\n\n TWELVE TIPS Twelve tips for giving feedback effectively in the clinical environment\n\n SUBHA RAMANI1 & SHARON K. KRACKOV2 1Harvard Medical School, USA, 2Albany Medical College, USA\n\nAbstract\n\nBackground: Feedback is an essential element of the educational process for clinical trainees. Performance-based feedback enables good habits to be reinforced and faulty ones to be corrected. Despite its importance, most trainees feel that they do not receive adequate feedback and if they do, the process is not effective. Aims and methods: The authors reviewed the literature on feedback and present the following 12 tips for clinical teachers to provide effective feedback to undergraduate and graduate medical trainees. In most of the tips, the focus is the individual teacher in clinical settings, although som

In [29]:
print("Available documents:\n")
for i, doc in enumerate(normalized_docs):
    print(f"{i} -> {doc['source']} ({doc['char_count']} chars) [{doc['source_method']}]")

doc_index = 5
preview_chars = 3000

doc = normalized_docs[doc_index]

print("\n" + "=" * 80)
print("SOURCE:", doc["source"])
print("TYPE:", doc["file_type"])
print("METHOD:", doc["source_method"])
print("CHAR COUNT:", doc["char_count"])
print("=" * 80)
print(doc["text"][:preview_chars])
print("\n--- END OF PREVIEW ---")

Available documents:

0 -> Ramani_Krackov_2012.pdf (23848 chars) [pymupdf]
1 -> Van de Ridder_Wijnen-Meijer_2023.pdf (16816 chars) [pymupdf]
2 -> Van de Ridder 2023_Pendletons Rules a Feedback Method.pdf (16816 chars) [pymupdf]
3 -> Hewson_Little_1998_giving feedback in medical education.pdf (29113 chars) [pymupdf]
4 -> Hattie and Timperley_2007.pdf (108656 chars) [pymupdf]
5 -> 2026-03-12_Allgemeines_Feedbackregeln.docx (3421 chars) [docling]
6 -> Fabry_Handbuch Feedback_Lehrende.pdf (64020 chars) [pymupdf]
7 -> Gigante-J_Getting-Beyond-Good-Job_How-to-Give-Effective-Feedback_Pediatrics-2011.pdf (15818 chars) [pymupdf]
8 -> Wisniewsi_Zierer_Hattie_2020.pdf (65180 chars) [pymupdf]
9 -> MEDIC_FLYER_Mentoring_E11_220630_Ansicht.pdf (6764 chars) [pymupdf]
10 -> MEDiC_2023_Leitfaden_1-1_Mentoring-aktuell.pdf (25112 chars) [pymupdf]
11 -> Kunz et al_2019.pdf (63753 chars) [pymupdf]
12 -> Poster_Feedback_MITZ.pdf (1214 chars) [pymupdf]
13 -> Hillienhof_2018.pdf (4041 chars) [pymupdf]
14 -> T

In [30]:
print(doc["text"][:preview_chars])

- Allgemeines - Dieser online-Selbstlern-Kurs "Feedback in der Lehre - Basics" gibt Ihnen eine ersten Überblick über: - **die Bedeutung von Feedback** - **Feedbackregeln** - **Feedbackmethoden** - **Tools** die an der Medizinischen Fakultät Dresden angeboten werden um Feedback zu Fördern.

Dieser Kurs kann allein oder in Kombination mit der [online-synchronen Feedbackschulung](https://tu-dresden.de/med/mf/carl/zentrum/copy_of_stabsstelle-didaktik-lehrforschung/didaktik-ordner/medizindidaktik-1/feedbackschulung-fuer-das-pj) genutzt werden.

- [Die Bedeutung von Feedback](https://elearning.med.tu-dresden.de/moodle/course/section.php?id=5916) - **Wirkung des eigenen Handelns** auf andere **kennenlernen** - „ **Blinde Flecken** “ in der Eigenwahrnehmung **aufdecken** - **Optimierung des Handelns** bezogen auf Effizienz, Patientensicherheit, zwischenmenschliche Komponenten, ... - **Förderung der Selbstreflexionsfähigkeit**

- [Feedbackregeln](https://elearning.med.tu-dresden.de/moodle/cours

In [31]:
from pathlib import Path
import zipfile
import os
import shutil

def extract_docx_images(docx_path, output_dir="docx_image_debug"):
    """
    Extract embedded images from a .docx file.
    A DOCX file is a zip archive, and images are usually stored in word/media/.
    """
    docx_path = Path(docx_path)
    output_dir = Path(output_dir) / docx_path.stem
    output_dir.mkdir(parents=True, exist_ok=True)

    extracted_files = []

    with zipfile.ZipFile(docx_path, "r") as z:
        for member in z.namelist():
            if member.startswith("word/media/"):
                filename = Path(member).name
                target_path = output_dir / filename

                with z.open(member) as src, open(target_path, "wb") as dst:
                    shutil.copyfileobj(src, dst)

                extracted_files.append(str(target_path))

    return extracted_files

In [32]:
doc_index = 5  # change to the DOCX you want to inspect
docx_path = normalized_docs[doc_index]["path"]

image_files = extract_docx_images(docx_path)

print(f"Extracted {len(image_files)} images from {docx_path}\n")
for img in image_files:
    print(img)

Extracted 4 images from DataSource/2026-03-12_Allgemeines_Feedbackregeln.docx

docx_image_debug/2026-03-12_Allgemeines_Feedbackregeln/image1.png
docx_image_debug/2026-03-12_Allgemeines_Feedbackregeln/image2.png
docx_image_debug/2026-03-12_Allgemeines_Feedbackregeln/image3.png
docx_image_debug/2026-03-12_Allgemeines_Feedbackregeln/image4.png


In [33]:
# %pip install pytesseract pillow

In [35]:
# OCR setup note:For German language support, you may need to install the Tesseract language pack:
# brew install tesseract-lang
# tesseract --list-langs

In [ ]:
import pytesseract
from PIL import Image
import re

def ocr_images(image_paths):
    """
    Run OCR on images and clean common OCR artifacts.
    """
    ocr_results = []

    for img_path in image_paths:
        try:
            img = Image.open(img_path)

            # OCR extraction
            text = pytesseract.image_to_string(img, lang="deu+eng")

            # Remove leading/trailing whitespace
            text = text.strip()

            # Fix hyphenated line breaks:
            text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

            # Replace remaining single line breaks with spaces
            text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

            if text:
                ocr_results.append({
                    "image": img_path,
                    "text": text
                })

        except Exception as e:
            print(f"OCR failed for {img_path}: {e}")

    return ocr_results

In [38]:
image_text_blocks = ocr_images(image_files)

print(f"OCR extracted text from {len(image_text_blocks)} images\n")

for block in image_text_blocks:
    print("=" * 70)
    print(block["image"])
    print()
    print(block["text"][:400])

OCR extracted text from 4 images

docx_image_debug/2026-03-12_Allgemeines_Feedbackregeln/image1.png

Konstruktives

Soll dem Empfänger heifen sich weiterzuentwickeln.

negatives . Positive Aspekte und Handlungsfelder aufzeigen. (Positives zuerst)

Direktes ... Den Empfänger direkt ansprechen.

Das Feedback sollte so zeitnah wie möglich erfolgen.
docx_image_debug/2026-03-12_Allgemeines_Feedbackregeln/image2.png

_—7 Nachricht

N Sender nr >> Empfänger N 12

(Schulz von Thun, Miteinander reden: 1, 2014)
docx_image_debug/2026-03-12_Allgemeines_Feedbackregeln/image3.png

1. Wahrnehmung

2. Wirkung

3. Wunsch/Empfehlung

> so konkret & objektiv wie möglich [&) (am besten mit Beispielen, die Sie sich vermerkt haben) Ich habe gesehen ... > keine Interpretation des Wahrgenommenen (Sie waren desinteressiert/unaufmerksam, Sie haben es nicht verstanden. A

> Wie hat die wahrgenommene Situation auf Sie gewirkt? oder auch © Wie hätten Sie sich in der Situation gefühlt, wen
docx_image_debug/2026-03-